In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]="platform"
os.environ["JAX_ENABLE_X64"]="false"

import multiprocessing
os.environ["XLA_FLAGS"]="--xla_force_host_platform_device_count={}".format(multiprocessing.cpu_count())
# os.environ["XLA_FLAGS"]="--xla_cpu_multi_thread_eigen=true intra_op_parallelism_threads=16"
import jax
# print(multiprocessing.cpu_count())
# print(jax.device_count())



from sampling import *
from wavefunction import *
from observables import *
from training import *
from utils import *
import time

from plotly import graph_objects as go



fig = go.FigureWidget()

# Energy mean
fig.add_trace(
    go.Scatter(
        x=[], y=[],
        mode='lines+markers',
        name='Energy',
        line=dict(color='blue')
    )
)

# Upper uncertainty bound: E + σ
fig.add_trace(
    go.Scatter(
        x=[], y=[],
        mode='lines',
        line=dict(width=0),
        name='Energy + σ',
        showlegend=False
    )
)

# Lower uncertainty bound: E - σ (filled to previous trace)
fig.add_trace(
    go.Scatter(
        x=[], y=[],
        mode='lines',
        fill='tonexty',
        fillcolor='rgba(0, 0, 255, 0.2)',
        line=dict(width=0),
        name='Energy ± σ'
    )
)

fig.update_layout(
    title='Energy with Uncertainty Band over Training Steps',
    xaxis_title='Training Step',
    yaxis_title='Energy'
)


In [ ]:
N = 10
eta = 1
g = 1.0
N_CORES = 16

x = np.random.normal(size=(N, 3))
model = MLP_TI(hidden_sizes=(3*N, 3*N))

params = model.init(jax.random.PRNGKey(int(time.time())), x)
y = model.apply(params,x)

@jit
def psi(params, config):
    return jnp.exp(-model.apply(params, config))

In [ ]:
sampler = Sampler(psi, (N, 3))

nchains = N_CORES
pos_initials = [jnp.zeros((N, 3)) for _ in range(nchains)]
seeds = jnp.arange(nchains)
var = .25 # tune this

samples, acc_rate = sampler.run_many_chains(params, nchains**4//nchains, 1000, 3, var, pos_initials, seeds)
print(f"Acceptance rate: {acc_rate}")
print(samples.shape)

In [ ]:
# show fig
fig.show()

In [ ]:
init_params = params
lr = 1e-2
MC_options= {
    "num_samples": 16**3,
    "thermalization": 1000,
    "skip": 2,
    "var": var,
    "nchains": 16,
    "seeds": [int(time.time()) + i for i in range(16)], # current time plus the index of the chain
    # "pos_initials": [jnp.zeros((N, 3)) for _ in range(nchains)],
    "pos_initials": [jax.random.normal(jax.random.PRNGKey(i + int(time.time())), shape=(N, 3)) for i in range(nchains)],
    "chain": True,
}
steps = 500

resultsa = train((init_params, [],[]), model, eta, g, sampler, MC_options, steps, lr, fig=fig)

In [ ]:
var = .3

samples, acc_rate = sampler.run_many_chains(resultsa[0], nchains**4//nchains, 1000, 3, var, pos_initials, seeds)
print(f"Acceptance rate: {acc_rate}")

In [ ]:
lr = 1e-3
MC_options= {
    "num_samples": 16**3,
    "thermalization": 1000,
    "skip": 2,
    "var": var,
    "nchains": 16,
    "seeds": [int(time.time()) + i for i in range(16)], # current time plus the index of the chain
    # "pos_initials": [jnp.zeros((N, 3)) for _ in range(nchains)],
    "pos_initials": [jax.random.normal(jax.random.PRNGKey(i + int(time.time())), shape=(N, 3)) for i in range(nchains)],
    "chain": True,
}
steps = 1000
resultsb = train(resultsa, model, eta, g, sampler, MC_options, steps, lr, fig=fig)

In [ ]:
var = 1

samples, acc_rate = sampler.run_many_chains(resultsb[0], nchains**4//nchains, 1000, 3, var, pos_initials, seeds)
print(f"Acceptance rate: {acc_rate}")

In [ ]:
lr = 1e-3
MC_options= {
    "num_samples": 16**4,
    "thermalization": 100,
    "skip": 2,
    "var": var,
    "nchains": 16,
    "seeds": [int(time.time()) + i for i in range(16)], # current time plus the index of the chain
    # "pos_initials": [jnp.zeros((N, 3)) for _ in range(nchains)],
    "pos_initials": [jax.random.normal(jax.random.PRNGKey(i + int(time.time())), shape=(N, 3)) for i in range(nchains)],
    "chain": True,
}
steps = 100
resultsc = train(resultsb, model, eta, g, sampler, MC_options, steps, lr, fig=fig)

In [ ]:
# save the parameters to a file
np.save(f"data/L_{N}_g_{g}_params.npy", resultsc[0])
# load them
# params = np.load("params.npy", allow_pickle=True).item()


In [ ]:
params = np.load(f"data/L_{N}_g_{g}_params.npy", allow_pickle=True).item()
pos_initials = [jnp.zeros((N, 3)) for _ in range(nchains)]
seeds = [int(time.time()) + i for i in range(16)]
var = .6

samples, acc_rate = sampler.run_many_chains(params, nchains**5//nchains, 1000, 3, var, pos_initials, seeds)
print(f"Acceptance rate: {acc_rate}")



In [ ]:
import jax.numpy.linalg as linalg
avgs = jnp.mean(samples, axis=0)
print(avgs.shape)
print(linalg.norm(avgs, axis=1))

In [ ]:
# compute the correlation functions
C = nx_n0(samples)

print(C)
# plot this with plotly
fig_c = go.Figure()
fig_c.add_trace(go.Scatter(x=np.arange(len(C)), y=C, mode='lines+markers'))
fig_c.update_layout(title='Correlation Function C(n)', xaxis_title='n', yaxis_title='C(n)')
fig_c.show()

In [ ]:
from scipy.optimize import curve_fit
# Define the system size based on your correlation array
L = len(C)

def exp_decay_pbc_standard(n, A, xi):
    # return A * (np.exp(-n / xi) + np.exp(-(L - n) / xi))
    return A * (np.exp(-n / xi))/(n/xi)

# Create an array of indices from 1 to L-1 (dropping n=0)
n_data = np.arange(1, L-1)

# Slice the correlation data to match (dropping the first element)
C_data = C[1:L-1]
# C_data = C

popt, pcov = curve_fit(exp_decay_pbc_standard, n_data, C_data, p0=[C_data[1], 1.0], maxfev=10000)

A_fit, xi_fit = popt
print(f"Fitted parameters: A = {A_fit:.4f}, xi = {xi_fit:.4f}")
# print the uncertainties (standard deviations) of the fitted parameters
perr = np.sqrt(np.diag(pcov))
print(f"Parameter uncertainties: σ_A = {perr[0]:.4f}, σ_xi = {perr[1]:.4f}")

In [ ]:
# plot the original data an0 the fitted curve
fig_fit = go.Figure()
fig_fit.add_trace(go.Scatter(x=n_data, y=C_data, mode='markers', name='Data'))
n_fit = np.linspace(0, L-1, 100)
C_fit = exp_decay_pbc_standard(n_fit, *popt)
fig_fit.add_trace(go.Scatter(x=n_fit, y=C_fit, mode='lines', name='Fitted Curve'))
fig_fit.update_layout(title='Correlation Function Fit', xaxis_title='n', yaxis_title='C(n)')
fig_fit.show()